# Roman CS Analysis — Baseline + Transfer Learning

Plots Taka scratch, Mead2020 scratch, and Taka→Mead TL models on the same axes.
Toggle groups and sizes in the CONFIG cell.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
import torch
import h5py
import sys
from pathlib import Path

BASE = Path('/home/grads/tmp/roman_cs_baseline')
sys.path.insert(0, str(BASE))
from emulator import ResCNN

# --------------------------------------------------------------------------
# Which groups to load/plot
# --------------------------------------------------------------------------
INCLUDE_TAKA = True
INCLUDE_MEAD = True
INCLUDE_TL   = True

TAKA_SIZES = [100_000, 250_000, 500_000, 1_000_000]
MEAD_SIZES = [10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]
TL_SIZES   = [10_000, 25_000, 50_000]

# --------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------
_TAKA_TEST_DV  = Path('/home/grads/backup/mltraining/yijie/roman_3x2_lcdm_taka/roman_real_lcdm_b_taka_test_datavectors.npy')
_TAKA_TEST_PAR = Path('/home/grads/backup/mltraining/yijie/roman_3x2_lcdm_taka/roman_real_lcdm_b_taka_test_parameters.txt')
_MEAD_TEST_DV  = Path('/home/grads/backup/mltraining/yijie/roman_3x2_lcdm/test/roman_real_lcdm_b_test_datavectors.npy')
_MEAD_TEST_PAR = Path('/home/grads/backup/mltraining/yijie/roman_3x2_lcdm/test/roman_real_lcdm_b_test_parameters.txt')


_CONFIGS = {
    'taka': {
        'model_dir':    BASE / 'output_taka' / 'models',
        'model_prefix': 'roman_cs_lcdm_taka',
        'log_file':     BASE / 'sweep_taka.log',
        'test_dv':      _TAKA_TEST_DV,
        'test_par':     _TAKA_TEST_PAR,
        'n_dv':         1080,
        'sizes':        TAKA_SIZES,
        'shared_h5':    None,
    },
    'mead': {
        'model_dir':    BASE / 'output_mead' / 'models',
        'model_prefix': 'roman_cs_lcdm_mead',
        # 10k–50k runs are in sweep_mead_small.log; 100k–1M in sweep_mead.log
        'log_files':    [BASE / 'sweep_mead_small.log', BASE / 'sweep_mead.log'],
        'log_file':     None,   # unused directly; see log_files
        'test_dv':      _MEAD_TEST_DV,
        'test_par':     _MEAD_TEST_PAR,
        'n_dv':         1080,
        'sizes':        MEAD_SIZES,
        'shared_h5':    None,
    },
    'tl': {
        'model_dir':    BASE / 'output_tl' / 'models',
        'model_prefix': 'roman_cs_lcdm_taka2mead_TL',
        'log_file':     None,   # no sweep_tl.log yet; update path once it exists
        'test_dv':      _MEAD_TEST_DV,  # TL evaluated on mead test data
        'test_par':     _MEAD_TEST_PAR,
        'n_dv':         1080,
        'sizes':        TL_SIZES,
        # Each TL run saves its own .h5 containing the pretrained taka normalization.
        # shared_h5=None means we use that per-run h5 — correct.
        'shared_h5':    None,
    },
}

OUTPUT_DIR = BASE / 'output_combined'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Config loaded.')
for g, c in _CONFIGS.items():
    print(f'  {g}: model_dir exists={c["model_dir"].exists()}')


In [ ]:
# ============================================================================
# PLOT SETTINGS
# ============================================================================
plt.rcParams['mathtext.fontset']     = 'stix'
plt.rcParams['font.family']          = 'STIXGeneral'
plt.rcParams['xtick.bottom']         = True
plt.rcParams['xtick.top']            = False
plt.rcParams['ytick.right']          = False
plt.rcParams['axes.edgecolor']       = 'black'
plt.rcParams['axes.linewidth']       = '1.0'
plt.rcParams['axes.grid']            = True
plt.rcParams['grid.linewidth']       = '0.0'
plt.rcParams['grid.alpha']           = '0.18'
plt.rcParams['grid.color']           = 'lightgray'
plt.rcParams['legend.labelspacing']  = 0.77
plt.rcParams['savefig.bbox']         = 'tight'
plt.rcParams['savefig.format']       = 'pdf'
plt.rcParams['text.usetex']          = True
plt.rcParams['text.latex.preamble']  = r'\usepackage{tipa}'
plt.rcParams.update({
    'font.size':        18,
    'axes.labelsize':   30,
    'axes.titlesize':   34,
    'xtick.labelsize':  26,
    'ytick.labelsize':  26,
    'legend.fontsize':  20,
    'figure.titlesize': 38,
})

# Sequential colormaps per group — avoids very light shades
_PALETTES = {'taka': plt.cm.Oranges, 'mead': plt.cm.Blues, 'tl': plt.cm.Purples}

def get_color(group, idx, total):
    frac = 0.4 + 0.5 * idx / max(total - 1, 1)
    return _PALETTES[group](frac)

def fmt_n(n):
    """Return a plain string for N — caller wraps in math mode if needed."""
    if n >= 1_000_000: return r'10^{6}'
    if n >= 1_000:     return f'{n//1000}\\mathrm{{k}}'
    return str(n)

GROUP_LABELS = {
    'taka': r'Taka (scratch)',
    'mead': r'Mead (scratch)',
    'tl':   r'TL Taka$\to$Mead',
}
GROUP_STYLES = {
    'taka': dict(color='#F18F01', marker='o', linestyle='-'),
    'mead': dict(color='#2E86AB', marker='s', linestyle='-'),
    'tl':   dict(color='#A23B72', marker='^', linestyle='--'),
}


In [ ]:
# ============================================================================
# LOADING — parse log first, then run inference for full Δχ² arrays
# ============================================================================

def parse_log(log_file_or_list):
    """
    Returns {n_train: {'mean', 'median', 'n_outliers'}} or {} if no log.
    Accepts a single path or a list of paths (results merged; later files win on overlap).
    """
    if log_file_or_list is None:
        return {}
    log_files = log_file_or_list if isinstance(log_file_or_list, list) else [log_file_or_list]
    parsed = {}
    for log_file in log_files:
        if log_file is None or not Path(log_file).exists():
            continue
        current_n = None
        with open(log_file, 'r') as f:
            for line in f:
                if 'N_train =' in line:
                    current_n = int(line.strip().split('N_train = ')[1])
                    parsed[current_n] = {}
                if current_n is None:
                    continue
                if 'Mean   Delta Chi2' in line:
                    parsed[current_n]['mean']       = float(line.strip().split('= ')[1])
                if 'Median Delta Chi2' in line:
                    parsed[current_n]['median']     = float(line.strip().split('= ')[1])
                if 'N points with Chi2 > 0.2' in line:
                    parsed[current_n]['n_outliers'] = int(line.strip().split(': ')[1])
    return parsed


def run_inference(group_key, n_train):
    """Load a ResCNN and return per-sample Δχ² array, or None if files missing."""
    cfg     = _CONFIGS[group_key]
    n_dv    = cfg['n_dv']
    pt_path = cfg['model_dir'] / f"{cfg['model_prefix']}_N{n_train}.pt"
    h5_path = cfg['shared_h5'] or (cfg['model_dir'] / f"{cfg['model_prefix']}_N{n_train}.h5")

    if not pt_path.exists():
        print(f'    [skip] not found: {pt_path.name}')
        return None
    if not h5_path.exists():
        print(f'    [skip] h5 not found: {h5_path}')
        return None

    with h5py.File(h5_path, 'r') as f:
        samples_mean = torch.tensor(f['sample_mean'][:], dtype=torch.float32)
        samples_std  = torch.tensor(f['sample_std'][:],  dtype=torch.float32)
        dv_evals     = torch.tensor(f['dv_evals'][:],    dtype=torch.float32)
        dv_evecs     = torch.tensor(f['dv_evecs'][:],    dtype=torch.float32)
        dv_fid       = torch.tensor(f['dv_fid'][:],      dtype=torch.float32)
        train_params = [p.decode() if isinstance(p, bytes) else p
                        for p in f['train_params'][:]]

    header = np.array(open(cfg['test_par']).readline().split(' ')[1:])
    header[-1] = header[-1].strip()
    idxs = [np.where(header == p)[0][0] for p in train_params]

    x_test = torch.tensor(np.loadtxt(cfg['test_par'])[:, idxs], dtype=torch.float32)
    y_test = torch.tensor(np.load(cfg['test_dv'])[:, :n_dv],    dtype=torch.float32)

    x_test      = (x_test - samples_mean) / samples_std
    y_test_norm = (y_test - dv_fid) @ dv_evecs / dv_evals

    model = ResCNN(len(train_params), n_dv, 256, 576, 21, 1)
    model.load_state_dict(torch.load(pt_path, map_location='cpu'))
    model.eval()

    with torch.no_grad():
        diff = y_test_norm - model(x_test)
    return (diff ** 2).sum(dim=1).numpy()


# --------------------------------------------------------------------------
# Main loop — log provides a sanity check; inference gives the full array
# --------------------------------------------------------------------------
results     = {}   # {group: {n: np.ndarray}}
log_results = {}   # {group: {n: {'median', 'mean', 'n_outliers'}}}

groups_to_load = [g for g, flag in [('taka', INCLUDE_TAKA),
                                     ('mead', INCLUDE_MEAD),
                                     ('tl',   INCLUDE_TL)] if flag]

for group in groups_to_load:
    results[group]     = {}
    log_results[group] = parse_log(_CONFIGS[group].get('log_files', _CONFIGS[group]['log_file']))
    print(f'\n=== {group.upper()} ===')
    if log_results[group]:
        print(f'  Log: {sorted(log_results[group].keys())} sizes available')

    for n in _CONFIGS[group]['sizes']:
        in_log = n in log_results[group]
        log_med = log_results[group][n].get('median', float('nan')) if in_log else float('nan')
        src = f'log median={log_med:.4f}, running inference...' if in_log else 'not in log, running inference...'
        print(f'  N={n:>9,}: {src}')

        dc2 = run_inference(group, n)
        if dc2 is not None:
            results[group][n] = dc2
            inf_med = np.median(dc2)
            note = ''
            if in_log and not np.isnan(log_med):
                match = '✓' if abs(inf_med - log_med) < 0.01 else '⚠ MISMATCH'
                note  = f'  (log={log_med:.4f} {match})'
            print(f'    → median={inf_med:.4f}, f>0.2={np.mean(dc2>0.2):.3f}{note}')

print('\nDone.')


In [ ]:
# ============================================================================
# PLOT 1 — All groups, all sizes
# ============================================================================

# Override to show only a subset per group (None = all loaded sizes)
TAKA_PLOT_SIZES = None   # e.g. [100_000, 1_000_000]
MEAD_PLOT_SIZES = None
TL_PLOT_SIZES   = None

_overrides = {'taka': TAKA_PLOT_SIZES, 'mead': MEAD_PLOT_SIZES, 'tl': TL_PLOT_SIZES}

def _sizes_to_plot(group):
    available = sorted(results.get(group, {}).keys())
    override  = _overrides.get(group)
    return [n for n in override if n in results.get(group, {})] if override else available


def plot_combined_chi2(save=False):
    fig, ax = plt.subplots(figsize=(13, 7))
    bins = np.logspace(-4, 2, 60)

    for group in ['taka', 'mead', 'tl']:
        sizes = _sizes_to_plot(group)
        for idx, n in enumerate(sizes):
            color = get_color(group, idx, len(sizes))
            ax.hist(results[group][n], bins=bins, alpha=0.55,
                    color=color, edgecolor='black', linewidth=0.4,
                    label=rf'{GROUP_LABELS[group]}, $N={fmt_n(n)}$')

    ax.axvline(0.2, color='red', linestyle='--', linewidth=2.5, alpha=0.85,
               label=r'$\Delta\chi^2 = 0.2$')
    ax.set_xscale('log')
    ax.set_xlabel(r'$\Delta\chi^2$')
    ax.set_ylabel(r'Count')
    ax.set_title(r'Roman CS $\Delta\chi^2$ Distributions')
    ax.legend(fontsize=15, ncol=2)

    if save:
        fpath = OUTPUT_DIR / 'chi2_distributions_combined.pdf'
        plt.savefig(fpath, bbox_inches='tight', dpi=300, pad_inches=0.05)
        print(f'Saved: {fpath}')
    plt.show()

plot_combined_chi2()


In [ ]:
# ============================================================================
# PLOT 2 — TL comparison: N=ANCHOR_N Taka + Mead scratch vs all TL sizes
# ============================================================================
ANCHOR_N = 100_000   # which scratch N to use as the comparison baseline


def plot_tl_comparison(save=False):
    fig, ax = plt.subplots(figsize=(13, 7))
    bins = np.logspace(-4, 2, 60)

    # Scratch anchor models
    for group in ['taka', 'mead']:
        if group not in results or ANCHOR_N not in results[group]:
            print(f'  [skip] {group} N={ANCHOR_N} not loaded')
            continue
        color = GROUP_STYLES[group]['color']
        label = rf'{GROUP_LABELS[group]}, $N={fmt_n(ANCHOR_N)}$'
        ax.hist(results[group][ANCHOR_N], bins=bins, alpha=0.65,
                color=color, edgecolor='black', linewidth=0.5,
                label=label, zorder=3)

    # TL fine-tune sizes
    tl_sizes = sorted(results.get('tl', {}).keys())
    for idx, n in enumerate(tl_sizes):
        color = get_color('tl', idx, len(tl_sizes))
        ax.hist(results['tl'][n], bins=bins, alpha=0.55,
                color=color, edgecolor='black', linewidth=0.4,
                label=rf'TL Taka$\to$Mead, $N={fmt_n(n)}$')

    ax.axvline(0.2, color='red', linestyle='--', linewidth=2.5, alpha=0.85,
               label=r'$\Delta\chi^2 = 0.2$')
    ax.set_xscale('log')
    ax.set_xlabel(r'$\Delta\chi^2$')
    ax.set_ylabel(r'Count')
    ax.set_title(rf'TL vs Scratch Baselines ($N_{{\rm anchor}} = {fmt_n(ANCHOR_N)}$)')
    ax.legend(fontsize=16)

    if save:
        fpath = OUTPUT_DIR / f'chi2_tl_vs_scratch_N{ANCHOR_N}.pdf'
        plt.savefig(fpath, bbox_inches='tight', dpi=300, pad_inches=0.05)
        print(f'Saved: {fpath}')
    plt.show()

plot_tl_comparison()


In [ ]:
# ============================================================================
# PLOT 3 — Scaling: median Δχ² and outlier fraction vs N_train
# ============================================================================

def plot_scaling(save=False):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for group, res in results.items():
        if not res:
            continue
        sizes   = sorted(res.keys())
        sizes_k = [n / 1000 for n in sizes]
        medians = [np.median(res[n]) for n in sizes]
        frac    = [np.mean(res[n] > 0.2) for n in sizes]
        sty     = GROUP_STYLES[group]
        kw = dict(linewidth=2.5, markersize=9, label=GROUP_LABELS[group],
                  color=sty['color'], marker=sty['marker'], linestyle=sty['linestyle'])
        axes[0].plot(sizes_k, medians, **kw)
        axes[1].plot(sizes_k, frac,    **kw)

    for ax in axes:
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel(r'$N_{\rm train}\;/\;1000$')
        ax.legend(fontsize=18)

    axes[0].axhline(0.2, color='red', linestyle='--', linewidth=2, alpha=0.8)
    axes[0].set_ylabel(r'Median $\Delta\chi^2$')
    axes[0].set_title(r'Median $\Delta\chi^2$ vs $N_{\rm train}$')

    axes[1].set_ylabel(r'$f(\Delta\chi^2 > 0.2)$')
    axes[1].set_title(r'Outlier fraction vs $N_{\rm train}$')

    plt.tight_layout()
    if save:
        fpath = OUTPUT_DIR / 'scaling_combined.pdf'
        plt.savefig(fpath, bbox_inches='tight', dpi=300, pad_inches=0.05)
        print(f'Saved: {fpath}')
    plt.show()

plot_scaling()


In [ ]:
# ============================================================================
# SUMMARY TABLE
# ============================================================================

print(f'{"Group":<8} {"N_train":>10} {"Median":>10} {"Mean":>10} {"f>0.2":>7}')
print('-' * 50)
for group in ['taka', 'mead', 'tl']:
    if group not in results:
        continue
    for n in sorted(results[group].keys()):
        dc2      = results[group][n]
        inf_med  = np.median(dc2)
        flag     = ''
        if n in log_results.get(group, {}):
            log_med = log_results[group][n].get('median', float('nan'))
            if not np.isnan(log_med) and abs(inf_med - log_med) > 0.01:
                flag = ' ⚠'
        print(f'{group:<8} {n:>10,} {inf_med:>10.4f} {np.mean(dc2):>10.4f} {np.mean(dc2>0.2):>7.3f}{flag}')
    print()
